# Updating Guardian Article Datasets

#### **IMPORT LIBRARIES**

In [ ]:
import numpy as np
import pandas as pd
import re
import requests
import time
import seaborn as sns
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

#### **EXTRACT GUARDIAN ARTICLES**

In [ ]:
# Extracting articles using the Guardian News API

def search_guardian_articles(GUARDIAN_KEY, query, min_results=50):
    print('1. Obtaining key article information')
    batches = []
    num_results = 0
    page_num = 1
    page = 'https://content.guardianapis.com/search?page=' + str(page_num) +  '&q=' + query + '%20%20&api-key=' + GUARDIAN_KEY
    
    # Makes requests until it has gathered at least min_results articles
    while num_results < min_results:
        response = requests.get(page,params={'search':query})
        time.sleep(1.1) # Respectful use of the API

        if response.status_code == 200:
            data = response.json()
            headlines = []
            dates = []
            sections = []
            web_urls = []

            # Stores key information
            for article in data['response']['results']:
                headlines.append(article['webTitle'])
                dates.append(article['webPublicationDate'])
                sections.append(article['sectionName'])
                web_urls.append(article['apiUrl'])

            # Adds information to a dataframe
            batch = pd.DataFrame(data={'headline':headlines,
                                            'date':dates,
                                            'section':sections,
                                            'url':web_urls})
            
            # Stores the dataframe in a batch for efficient handling
            batches.append(batch) 

            # Update values for next iteration
            num_results += len(data['response']['results'])
            page_num += 1
            page = 'https://content.guardianapis.com/search?page=' + str(page_num) +  '&q=' + query + '%20%20&api-key=' + GUARDIAN_KEY

        else:
            print('Error with API request.')
    
        print(f'Progress: {num_results} / {min_results}')

    df = pd.concat(batches,ignore_index=True) # combines all the batches at once
    df = df.drop_duplicates() # Removes any duplicate articles (rare)
    df = df.reset_index(drop=True)

    return df

def get_article_contents(GUARDIAN_KEY, df):
    print('\n2. Extracting article content')
    texts = []
    n_urls = len(df['url'])
    for i in range(n_urls):
        response = requests.get(url=df['url'].iloc[i],params={"api-key": GUARDIAN_KEY,"show-fields": "all"})
        data = response.json()
        texts.append(data['response']['content']['fields']["bodyText"])
        time.sleep(1)

        # Reports progress every 10 URLs
        if (i+1) % 10 == 0:
            print(f'Progress: {i+1} / {n_urls}')

    df['text'] = texts

    return df

In [ ]:
GUARDIAN_KEY = '' # Enter your Guardian API key here!
query = "Climate Change"
min_results = 600
df = search_guardian_articles(GUARDIAN_KEY, query, min_results)
df = get_article_contents(GUARDIAN_KEY, df)
docs = df['text'].to_list()
df.to_csv('guardian_articles.csv',encoding='utf-8-sig',index=False)

#### **READ GUARDIAN DATA**

In [240]:
df = pd.read_csv('csv_data/guardian_articles.csv',encoding='utf-8')
df = df.dropna()
docs = df['text'].to_list()
print(f'Your corpus consists of {len(docs)} documents.')
df.head()

Your corpus consists of 595 documents.


,headline,date,section,url,text
0,Donald Trump’s perverse policy on climate chan...,2026-01-29T17:32:10Z,US news,https://content.guardianapis.com/us-news/2026/...,"Jeremy Wallace, professor of China studies at ..."
1,Kemi Badenoch vows to repeal Climate Change Act,2025-10-02T05:00:40Z,Environment,https://content.guardianapis.com/environment/2...,Kemi Badenoch has vowed to repeal the Climate ...
2,Relentless sun and ruthless populists: how the...,2026-03-04T13:00:21Z,Life and style,https://content.guardianapis.com/lifeandstyle/...,After a diplomatic career spent in the war zon...
3,"Less snow, or more risk? What you need to know...",2026-02-22T09:00:33Z,World news,https://content.guardianapis.com/world/2026/fe...,Avalanches kill about 100 people in Europe eac...
4,Antarctic penguins have radically shifted thei...,2026-01-20T05:01:13Z,World news,https://content.guardianapis.com/world/2026/ja...,Penguins in Antarctica have radically shifted ...


#### **CALCULATE TOPICS**

In [5]:
# Embedding model

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embedding_model.encode(docs, show_progress_bar=True)

print(f"Embeddings shape: {embeddings.shape}")
print(f"Each document is now a vector of {embeddings.shape[1]} numbers")


# UMAP

umap_model = UMAP(
    n_neighbors=10,
    n_components=5,
    min_dist=0.0,
    metric='cosine',
    random_state=42
)

reduced_embeddings = umap_model.fit_transform(embeddings)

print(f"Before UMAP: {embeddings.shape}")
print(f"After UMAP:  {reduced_embeddings.shape}")


# hdbcan model

hdbscan_model = HDBSCAN(
    min_cluster_size=8, # Adjust according to corpus size
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

cluster_labels = hdbscan_model.fit_predict(reduced_embeddings)

n_topics = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
n_noise = np.sum(cluster_labels == -1)

print(f"Topics found: {n_topics}")
print(f"Documents assigned to noise (topic -1): {n_noise} ({100*n_noise/len(docs):.1f}%)")
print(f"Documents assigned to a topic: {len(docs) - n_noise}")


# Count vectoriser
vectorizer = CountVectorizer(stop_words='english')


# BERTopic
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer,
    language='english',
    verbose=True
)

topic_ids, topic_probabilities = topic_model.fit_transform(docs, embeddings=embeddings)
topic_top_terms = []

print(f"\nTopics found: {len(topic_model.get_topic_info()) - 1}")
print("\nTop terms per topic:")

for topic_id in sorted(topic_model.get_topics().keys()):
    if topic_id == -1:
        continue
    
    top_terms = [t[0] for t in topic_model.get_topic(topic_id)[:8]]
    topic_top_terms.append(top_terms)
    print(f"  Topic {topic_id:2d}: {', '.join(top_terms)}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Embeddings shape: (595, 384)
Each document is now a vector of 384 numbers


2026-04-09 14:00:08,552 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


Before UMAP: (595, 384)
After UMAP:  (595, 5)
Topics found: 21
Documents assigned to noise (topic -1): 181 (30.4%)
Documents assigned to a topic: 414


2026-04-09 14:00:09,550 - BERTopic - Dimensionality - Completed ✓
2026-04-09 14:00:09,551 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-09 14:00:09,575 - BERTopic - Cluster - Completed ✓
2026-04-09 14:00:09,594 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-09 14:00:10,066 - BERTopic - Representation - Completed ✓



Topics found: 21

Top terms per topic:
  Topic  0: uk, climate, finance, energy, government, countries, said, year
  Topic  1: climate, countries, cop30, cop, said, fossil, world, brazil
  Topic  2: ai, said, hearing, economics, just, google, work, fashion
  Topic  3: epa, climate, finding, endangerment, court, said, air, trump
  Topic  4: police, said, people, government, protest, port, justice, british
  Topic  5: food, farmers, uk, climate, year, prices, crop, said
  Topic  6: film, novel, films, helm, people, climate, world, says
  Topic  7: league, england, season, players, ball, game, just, team
  Topic  8: reform, party, said, uk, labour, starmer, bbc, farage
  Topic  9: ice, sea, heat, arctic, water, record, temperatures, climate
  Topic 10: children, school, schools, needs, send, changes, special, support
  Topic 11: climate, news, said, science, change, corp, energy, australia
  Topic 12: climate, icj, court, states, pacific, legal, opinion, international
  Topic 13: target,

#### **SELECT TOPICS**

In [286]:
# Inside the 'keep_topics' dictionary, add the topic number and give the topic a name for all topics you wish to keep.

climate_topics = {
    0:'UK Climate & Energy Policy',
    1:'COP30',
    3:'Trump - Environmental Protection Agency Rollbacks',
    5:'UK Food Security',
    6:'Climate in Film & Literature',
    9:'Arctic Ice Melt & Sea Level Rise',
    11:'Climate News Australia',
    12:'Climate Law from the International Court of Justice',
    13:'Australia 2035 Emissions Targets',
    15:'Wildfires',
    16:'Coral Reefs & Sea Turtles',
    17:'US Climate Policy',
    18:'Australian Liberal Coalition & Net Zero',
    20:'Extreme Weather Events'
}

topic_ids = []
climate_topic_top_terms = []
topic_names = []
for i in climate_topics.keys():
    topic_ids.append(i)
    climate_topic_top_terms.append(topic_top_terms[i])
    topic_names.append(climate_topics[i])

topics_df = pd.DataFrame(data={'topic_name':topic_names,'topic_top_terms':climate_topic_top_terms,'topic_id':topic_ids})

#### **SENTIMENT OF EACH ARTICLE & HEADLINE**

In [241]:
# Initialise Vader
vader = SentimentIntensityAnalyzer()

# Sentiment of each article & headline
article_sentiments = []
for doc in docs:
    article_sentiments.append(vader.polarity_scores(doc)['compound'])

headline_sentiments = []
for i in df['headline'].to_list():
    headline_sentiments.append(vader.polarity_scores(i)['compound'])

df['article_sentiments'] = article_sentiments
df['headline_sentiments'] = headline_sentiments

#### **TOPIC SENTIMENT BY ARTICLE & HEADLINE**

In [290]:
# Count the number of times each topic's words appear across all articles
# Also calculates a score for the article and headline sentiment of each topic

article_topic_sentiments = []
headline_topic_sentiments = []
for i in range(len(topics_df)): # For each topic
    topic_counts = []
    for doc in docs: # Each article
        words = re.findall(r"[A-Za-z]+", doc.lower())
        count = 0
        for topic_word in topic_top_terms[i]: # For each topic word
            for word in words: # for each article word
                if word == topic_word: # count if there's a match
                    count += 1
        topic_counts.append(count)
    
    column_name = 'counts_' + str(i)
    df[column_name] = topic_counts
    article_topic_sentiments.append((df[column_name] * df['article_sentiments']).mean())
    headline_topic_sentiments.append((df[column_name] * df['headline_sentiments']).mean())
    
topics_df['article_sentiment'] = article_topic_sentiments
topics_df['headline_sentiment'] = headline_topic_sentiments


topic_counts = []
for i in range(len(topics_df)):
    topic_column_name = 'counts_' + str(i)
    topic_counts.append(df[topic_column_name].sum())

topics_df['topic_count'] = topic_counts
topics_df.sort_values('article_sentiment',ascending=False)
topics_df.to_csv('csv_data/topic_data.csv',index=False)

#### **READ TOPIC DATA**

In [293]:
topics_df = pd.read_csv('csv_data/topic_data.csv',encoding='utf-8')
print(f'You have {len(topics_df)} topics stored to the csv.')
topics_df

You have 14 topics stored to the csv.


,topic_name,topic_top_terms,topic_id,article_sentiment,headline_sentiment,topic_count
0,UK Climate & Energy Policy,"['uk', 'climate', 'finance', 'energy', 'govern...",0,2.598328,-3.149025,12096
1,COP30,"['climate', 'countries', 'cop30', 'cop', 'said...",1,1.911090,-2.460270,10624
2,Trump - Environmental Protection Agency Rollbacks,"['epa', 'climate', 'finding', 'endangerment', ...",3,1.162371,-1.204871,5050
3,UK Food Security,"['food', 'farmers', 'uk', 'climate', 'year', '...",5,0.687412,-2.090154,9271
4,Climate in Film & Literature,"['film', 'novel', 'films', 'helm', 'people', '...",6,0.510837,-1.792033,6366
5,Arctic Ice Melt & Sea Level Rise,"['ice', 'sea', 'heat', 'arctic', 'water', 'rec...",9,1.166084,-2.736235,9814
6,Climate News Australia,"['climate', 'news', 'said', 'science', 'change...",11,1.824666,-1.574836,7598
7,Climate Law from the International Court of Ju...,"['climate', 'icj', 'court', 'states', 'pacific...",12,0.843986,-0.119419,1568
8,Australia 2035 Emissions Targets,"['target', 'australia', 'government', 'climate...",13,0.846414,-1.546160,4933
9,Wildfires,"['forest', 'fires', 'forests', 'trees', 'wildf...",15,0.208485,-1.308734,5513
